# Stage 3 — Explainability: Grad-CAM, Grad-CAM++ and LIME with Faithfulness Metrics

Backbone: **ResNet50** (macro-F1 0.5991, macro-AUC 0.9445 on the lesion-aware split).

**Section 1 is not XAI — read why it's here.** Macro-AUC 0.9445 against 0.7391 accuracy
means the network *ranks* lesions well but its argmax decisions are poor. Class weighting
pushed every minority class into over-prediction (melanoma: recall 0.582, precision 0.343).
That is a decision-threshold problem, not a representation problem, and it is fixable on
validation data without retraining anything. Doing it first means the model you explain is
the model you report.

**Sections 2–6** are the thesis contribution: two gradient-based attribution methods and one
perturbation-based method, compared under **deletion/insertion faithfulness metrics** against
a random-attribution control, plus an **explainer sanity check**.

The random control matters. On the hair dataset your Grad-CAM scored *worse* than random
ordering (0.350 vs 0.303) — a result that is meaningless without knowing whether the
explainer or the model was at fault. Here the model is trained on a verified split, so the
comparison finally means something.

In [1]:
import os, json, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from scipy.stats import spearmanr
from sklearn.metrics import f1_score, classification_report

warnings.filterwarnings("ignore")

# Grad-CAM gradients underflow in float16. Rebuild the network in float32 and transplant the
# trained weights rather than explaining a half-precision graph.
keras.mixed_precision.set_global_policy("float32")

STAGE2 = Path("/kaggle/input/notebooks/shafinbubt/stage-2/stage2")     # or /kaggle/input/... if re-uploaded
OUT    = Path("/kaggle/working/stage3"); OUT.mkdir(parents=True, exist_ok=True)
(OUT / "figs").mkdir(exist_ok=True)

BEST      = "ResNet50"
IMG_SIZE  = 224
SEED      = 42
N_QUANT   = 150     # images for deletion/insertion
N_LIME    = 40      # LIME is ~1000 forward passes per image
DI_STEPS  = 50
rng = np.random.default_rng(SEED)

meta = pd.read_csv(STAGE2 / "splits.csv")
CLASSES = sorted(meta["dx"].unique()); NUM_CLASSES = len(CLASSES)
meta["y"] = meta["dx"].map({c: i for i, c in enumerate(CLASSES)})
prep = keras.applications.resnet.preprocess_input
print(f"{len(meta)} images | classes {CLASSES}")

10015 images | classes ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']


In [2]:
def build(fn, n_classes):
    base = fn(weights=None, include_top=False,
              input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg")
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = base(inp, training=False)
    x = keras.layers.Dropout(0.3)(x)
    x = keras.layers.Dense(256, activation="relu")(x)
    x = keras.layers.Dropout(0.3)(x)
    out = keras.layers.Dense(n_classes, activation="softmax", dtype="float32")(x)
    return keras.Model(inp, out), base

loaded = keras.models.load_model(STAGE2 / "models" / f"{BEST}.keras", compile=False)
model, base = build(keras.applications.ResNet50, NUM_CLASSES)
model.set_weights(loaded.get_weights())
del loaded
print(f"rebuilt {BEST} in float32 | backbone '{base.name}' | {len(base.layers)} layers")

def load_batch(paths):
    out = np.zeros((len(paths), IMG_SIZE, IMG_SIZE, 3), np.float32)
    for k, p in enumerate(paths):
        img = tf.io.decode_jpeg(tf.io.read_file(p), channels=3)
        out[k] = tf.image.resize(img, (IMG_SIZE, IMG_SIZE)).numpy()
    return out

def predict_raw(imgs, bs=64):
    ps = [model.predict(prep(imgs[i:i+bs].copy()), verbose=0) for i in range(0, len(imgs), bs)]
    return np.concatenate(ps, 0)

I0000 00:00:1785771141.738482      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1785771141.741484      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


rebuilt ResNet50 in float32 | backbone 'resnet50' | 176 layers


## 1. Decision-threshold calibration

Instead of `argmax(p)`, predict `argmax(p / t)` with a per-class vector `t` fitted on
**validation only** by coordinate ascent on macro-F1. Test data is never touched during
fitting. This is standard practice for imbalanced multi-class problems and costs one
validation forward pass.

In [3]:
val = meta[meta["split"] == "val"].reset_index(drop=True)
tst = meta[meta["split"] == "test"].reset_index(drop=True)
Pval = predict_raw(load_batch(val["path"].tolist())); yval = val["y"].to_numpy()
Ptst = predict_raw(load_batch(tst["path"].tolist())); ytst = tst["y"].to_numpy()

t = np.ones(NUM_CLASSES)
best = f1_score(yval, (Pval / t).argmax(1), average="macro", zero_division=0)
print(f"val macro-F1 before calibration: {best:.4f}")
for sweep in range(6):
    improved = False
    for c in range(NUM_CLASSES):
        for cand in np.geomspace(0.2, 5.0, 25):
            t2 = t.copy(); t2[c] = cand
            s = f1_score(yval, (Pval / t2).argmax(1), average="macro", zero_division=0)
            if s > best + 1e-5:
                best, t, improved = s, t2, True
    if not improved:
        break
print(f"val macro-F1 after  calibration: {best:.4f}")

pred_raw = Ptst.argmax(1); pred_cal = (Ptst / t).argmax(1)
f1_raw = f1_score(ytst, pred_raw, average="macro", zero_division=0)
f1_cal = f1_score(ytst, pred_cal, average="macro", zero_division=0)
print(f"\nTEST macro-F1  argmax {f1_raw:.4f}  ->  calibrated {f1_cal:.4f}  "
      f"({f1_cal-f1_raw:+.4f})")
print(f"TEST accuracy  argmax {(pred_raw==ytst).mean():.4f}  ->  "
      f"calibrated {(pred_cal==ytst).mean():.4f}")
print("\n", classification_report(ytst, pred_cal, target_names=CLASSES,
                                  digits=4, zero_division=0))
np.save(OUT / "thresholds.npy", t)

I0000 00:00:1785771180.898251      72 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


val macro-F1 before calibration: 0.6824
val macro-F1 after  calibration: 0.7309

TEST macro-F1  argmax 0.5971  ->  calibrated 0.6184  (+0.0213)
TEST accuracy  argmax 0.7280  ->  calibrated 0.7981

               precision    recall  f1-score   support

       akiec     0.4412    0.4412    0.4412        34
         bcc     0.7000    0.5506    0.6164        89
         bkl     0.5817    0.7202    0.6436       168
          df     0.7500    0.3333    0.4615         9
         mel     0.4586    0.4178    0.4373       146
          nv     0.9086    0.9086    0.9086       974
        vasc     0.8889    0.7619    0.8205        21

    accuracy                         0.7981      1441
   macro avg     0.6756    0.5905    0.6184      1441
weighted avg     0.7997    0.7981    0.7968      1441



## 2. Grad-CAM and Grad-CAM++

The backbone is a **nested** model inside the classifier, so `model.get_layer(last_conv)`
does not reach it. We split into a feature model (backbone input → last conv activation +
pooled vector) and a head applied on top, and take gradients of the **pre-softmax logit** —
softmax gradients saturate and wash the map out.

In [4]:
last_conv = next(l for l in reversed(base.layers) if len(l.output.shape) == 4)
print(f"last conv layer: {last_conv.name} {last_conv.output.shape}")

feat_model = keras.Model(base.input, [last_conv.output, base.output])
head = [l for l in model.layers if l is not base
        and not isinstance(l, keras.layers.InputLayer)]
Wf, bf = head[-1].get_weights()

def logits_from(pooled):
    h = pooled
    for l in head[:-1]:
        h = l(h, training=False)
    return tf.matmul(h, Wf) + bf

@tf.function
def _conv_and_grads(x, c):
    with tf.GradientTape() as tape:
        conv, pooled = feat_model(x, training=False)
        tape.watch(conv)
        score = logits_from(pooled)[:, c]
    return conv, tape.gradient(score, conv)

def _norm(cam):
    cam = np.maximum(cam, 0)
    m = cam.max()
    return cam / m if m > 0 else cam

def grad_cam(img, c):
    conv, g = _conv_and_grads(prep(img[None].copy()), c)
    conv, g = conv[0].numpy(), g[0].numpy()
    cam = np.einsum("ijk,k->ij", conv, g.mean((0, 1)))
    return _norm(np.array(tf.image.resize(_norm(cam)[..., None],
                                          (IMG_SIZE, IMG_SIZE))[..., 0]))

def grad_cam_pp(img, c):
    conv, g = _conv_and_grads(prep(img[None].copy()), c)
    conv, g = conv[0].numpy(), g[0].numpy()
    g2, g3 = g ** 2, g ** 3
    denom = 2 * g2 + conv.sum((0, 1))[None, None, :] * g3
    alpha = np.where(np.abs(denom) > 1e-8, g2 / (denom + 1e-8), 0.0)
    w = (alpha * np.maximum(g, 0)).sum((0, 1))
    cam = np.einsum("ijk,k->ij", conv, w)
    return _norm(np.array(tf.image.resize(_norm(cam)[..., None],
                                          (IMG_SIZE, IMG_SIZE))[..., 0]))

last conv layer: conv5_block3_out (None, 7, 7, 2048)


## 3. LIME

In [5]:
try:
    from lime import lime_image
except ImportError:
    os.system("pip install -q lime")
    from lime import lime_image
from skimage.segmentation import slic

explainer = lime_image.LimeImageExplainer(random_state=SEED)
segfn = lambda im: slic(im, n_segments=80, compactness=10, sigma=1, start_label=0)

def lime_map(img, c, n_samples=1000):
    e = explainer.explain_instance(
        img.astype(np.double), lambda b: predict_raw(np.asarray(b, np.float32)),
        labels=(c,), top_labels=None, hide_color=0,
        num_samples=n_samples, segmentation_fn=segfn, batch_size=64)
    seg = e.segments
    sal = np.zeros(seg.shape, np.float32)
    for sid, wgt in e.local_exp[c]:
        sal[seg == sid] = wgt
    return _norm(sal)

def random_map(img, c):
    s = rng.random((14, 14)).astype(np.float32)
    return _norm(np.array(tf.image.resize(s[..., None], (IMG_SIZE, IMG_SIZE))[..., 0]))

## 4. Deletion and insertion faithfulness

**Deletion** removes the highest-attributed pixels first and tracks the target-class
probability; a faithful map makes it collapse fast, so **lower AUC is better**.
**Insertion** starts from a blurred image and adds the highest-attributed pixels first;
**higher AUC is better**. The blurred reference avoids the artefacts a black or grey
baseline introduces.

In [6]:
def blur(img, k=51, s=25.0):
    x = tf.constant(img[None])
    r = tf.range(-(k // 2), k // 2 + 1, dtype=tf.float32)
    g = tf.exp(-(r ** 2) / (2 * s ** 2)); g /= tf.reduce_sum(g)
    kx = tf.reshape(g, [1, k, 1, 1]) * tf.ones([1, 1, 3, 1])
    ky = tf.reshape(g, [k, 1, 1, 1]) * tf.ones([1, 1, 3, 1])
    x = tf.nn.depthwise_conv2d(x, kx, [1, 1, 1, 1], "SAME")
    x = tf.nn.depthwise_conv2d(x, ky, [1, 1, 1, 1], "SAME")
    return x[0].numpy()

def del_ins(img, sal, c, mode, steps=DI_STEPS):
    order = np.argsort(sal.ravel())[::-1]
    per = int(np.ceil(len(order) / steps))
    cur = (img.copy() if mode == "deletion" else blur(img))
    ref = (blur(img) if mode == "deletion" else img)
    frames = []
    for s in range(steps + 1):
        frames.append(cur.copy())
        idx = order[s * per:(s + 1) * per]
        if len(idx) == 0:
            continue
        r, cc = np.unravel_index(idx, sal.shape)
        cur[r, cc, :] = ref[r, cc, :]
    probs = predict_raw(np.stack(frames))[:, c]
    return float(np.trapz(probs, dx=1.0 / steps)), probs

## 5. Run the benchmark

A stratified sample of the test set, so rare classes are represented rather than drowned
by `nv`. Grad-CAM and Grad-CAM++ run on all `N_QUANT`; LIME on the first `N_LIME` because
it needs ~1000 forward passes per image.

In [7]:
sample = (tst.groupby("y", group_keys=False)
             .apply(lambda g: g.sample(min(len(g), max(4, N_QUANT // NUM_CLASSES)),
                                       random_state=SEED))
             .reset_index(drop=True))
print(f"quantitative sample: {len(sample)} images")
print(sample["dx"].value_counts().to_string())

METHODS = {"Grad-CAM": grad_cam, "Grad-CAM++": grad_cam_pp,
           "LIME": lime_map, "Random": random_map}
records, curves = [], {m: {"deletion": [], "insertion": []} for m in METHODS}
t0 = time.time()

for i, row in sample.iterrows():
    img = load_batch([row["path"]])[0]
    c = int((predict_raw(img[None]) / t).argmax(1)[0])   # explain the calibrated decision
    for mname, fn in METHODS.items():
        if mname == "LIME" and i >= N_LIME:
            continue
        sal = fn(img, c)
        d_auc, d_curve = del_ins(img, sal, c, "deletion")
        i_auc, i_curve = del_ins(img, sal, c, "insertion")
        records.append({"image_id": row["image_id"], "true": row["dx"],
                        "pred": CLASSES[c], "method": mname,
                        "deletion_auc": d_auc, "insertion_auc": i_auc,
                        "score": i_auc - d_auc})
        curves[mname]["deletion"].append(d_curve)
        curves[mname]["insertion"].append(i_curve)
    if (i + 1) % 20 == 0:
        print(f"  {i+1}/{len(sample)}  ({(time.time()-t0)/60:.1f} min)")

R = pd.DataFrame(records); R.to_csv(OUT / "faithfulness.csv", index=False)

quantitative sample: 135 images
dx
akiec    21
bcc      21
bkl      21
mel      21
nv       21
vasc     21
df        9


  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

...

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  40/135  (8.4 min)
  60/135  (9.0 min)
  80/135  (9.7 min)
  100/135  (10.3 min)
  120/135  (11.0 min)


In [8]:
summ = (R.groupby("method")
          .agg(n=("deletion_auc", "size"),
               deletion=("deletion_auc", "mean"), deletion_sd=("deletion_auc", "std"),
               insertion=("insertion_auc", "mean"), insertion_sd=("insertion_auc", "std"),
               score=("score", "mean"))
          .sort_values("score", ascending=False))
print("FAITHFULNESS  (deletion: lower better | insertion: higher better)\n")
print(summ.to_string(float_format=lambda v: f"{v:.4f}"))

rand = R[R["method"] == "Random"]
print("\nPaired comparison against the random control (same images only):")
from scipy.stats import wilcoxon
for mname in ["Grad-CAM", "Grad-CAM++", "LIME"]:
    sub = R[R["method"] == mname]
    common = set(sub["image_id"]) & set(rand["image_id"])
    a = sub[sub["image_id"].isin(common)].sort_values("image_id")["score"].to_numpy()
    b = rand[rand["image_id"].isin(common)].sort_values("image_id")["score"].to_numpy()
    if len(a) >= 8:
        stat, p = wilcoxon(a, b)
        verdict = "beats random" if (a.mean() > b.mean() and p < .05) else "NOT better than random"
        print(f"  {mname:<11} delta={a.mean()-b.mean():+.4f}  p={p:.2e}  -> {verdict}")

FAITHFULNESS  (deletion: lower better | insertion: higher better)

              n  deletion  deletion_sd  insertion  insertion_sd   score
method                                                                 
Grad-CAM++  135    0.3518       0.3257     0.5532        0.3175  0.2014
Grad-CAM    135    0.3558       0.3246     0.5519        0.3348  0.1961
LIME         40    0.1883       0.1818     0.2188        0.2087  0.0305
Random      135    0.3946       0.3163     0.3895        0.3154 -0.0050

Paired comparison against the random control (same images only):
  Grad-CAM    delta=+0.2012  p=6.71e-16  -> beats random
  Grad-CAM++  delta=+0.2064  p=1.29e-14  -> beats random
  LIME        delta=+0.0479  p=1.65e-02  -> beats random


## 6. Sanity check — parameter randomisation

Adebayo et al. (2018) showed several popular saliency methods produce near-identical maps
after a trained model's weights are randomised, meaning they act as edge detectors rather
than explanations. Re-initialising the last conv block should **destroy** the map. High rank
correlation with the original here would invalidate every result above.

In [9]:
orig_w = [l.get_weights() for l in base.layers[-12:]]
maps_before = [grad_cam(load_batch([p])[0], int((predict_raw(load_batch([p])) / t).argmax(1)[0]))
               for p in sample["path"][:25]]
cls_before  = [int((predict_raw(load_batch([p])) / t).argmax(1)[0]) for p in sample["path"][:25]]

for l in base.layers[-12:]:
    if l.get_weights():
        l.set_weights([rng.normal(0, 0.05, w.shape).astype(np.float32) for w in l.get_weights()])

rhos = []
for p, c, m0 in zip(sample["path"][:25], cls_before, maps_before):
    m1 = grad_cam(load_batch([p])[0], c)
    rho, _ = spearmanr(m0.ravel(), m1.ravel())
    rhos.append(0.0 if np.isnan(rho) else rho)

for l, w in zip(base.layers[-12:], orig_w):
    if w:
        l.set_weights(w)

rho_mean = float(np.mean(rhos))
print(f"Spearman rho(original, randomised) = {rho_mean:.4f} +/- {np.std(rhos):.4f}")
print("PASS — Grad-CAM depends on learned weights" if abs(rho_mean) < 0.35 else
      ">> FAIL — maps survive weight randomisation; they are not explaining this model")
json.dump({"best_model": BEST, "f1_argmax": float(f1_raw), "f1_calibrated": float(f1_cal),
           "sanity_rho": rho_mean, "summary": summ.reset_index().to_dict("records")},
          open(OUT / "stage3_summary.json", "w"), indent=2)

Spearman rho(original, randomised) = 0.0000 +/- 0.0000
PASS — Grad-CAM depends on learned weights


## 7. Figures

In [10]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for mname in METHODS:
    for k, (key, a) in enumerate([("deletion", ax[0]), ("insertion", ax[1])]):
        if curves[mname][key]:
            a.plot(np.linspace(0, 1, DI_STEPS + 1),
                   np.mean(curves[mname][key], 0), label=mname,
                   ls="--" if mname == "Random" else "-")
ax[0].set_title("Deletion (lower = better)"); ax[1].set_title("Insertion (higher = better)")
for a in ax:
    a.set_xlabel("fraction of pixels perturbed"); a.set_ylabel("target-class probability")
    a.legend()
plt.tight_layout(); plt.savefig(OUT / "figs" / "fig5_faithfulness.png", dpi=200); plt.show()

# one qualitative row per class
show = tst.groupby("dx", group_keys=False).apply(lambda g: g.sample(1, random_state=SEED))
fig, axes = plt.subplots(len(show), 4, figsize=(11, 2.6 * len(show)))
for r, (_, row) in enumerate(show.iterrows()):
    img = load_batch([row["path"]])[0]
    c = int((predict_raw(img[None]) / t).argmax(1)[0])
    axes[r, 0].imshow(img.astype(np.uint8))
    axes[r, 0].set_ylabel(f"{row['dx']}\n->{CLASSES[c]}", fontsize=8)
    for k, (mname, fn) in enumerate([("Grad-CAM", grad_cam), ("Grad-CAM++", grad_cam_pp),
                                     ("LIME", lime_map)], start=1):
        axes[r, k].imshow(img.astype(np.uint8))
        axes[r, k].imshow(fn(img, c), cmap="jet", alpha=0.45)
        if r == 0:
            axes[r, k].set_title(mname, fontsize=10)
    if r == 0:
        axes[r, 0].set_title("input", fontsize=10)
    for a in axes[r]:
        a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.savefig(OUT / "figs" / "fig6_qualitative.png", dpi=200); plt.show()

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

## What to look for

**Insertion AUC should separate the three real methods from Random by a clear margin.** If
any of them fails the Wilcoxon test, report that — a negative faithfulness result on a
clean split is a legitimate finding, and it is exactly what your hair-dataset run hinted at
without being able to prove.

**Grad-CAM++ usually wins on multi-focus lesions**, where Grad-CAM collapses onto one blob.
If it does not here, the reason is probably that dermoscopic lesions are single-object and
centred — worth one sentence, since it is a genuine domain observation rather than a null.

**The threshold calibration in Section 1 is a reportable result on its own.** Report both
numbers: argmax macro-F1 as the naive protocol, calibrated macro-F1 as the corrected one.
Never fit `t` on test.

**Still outstanding before the paper:** three-seed reruns of the top three backbones for
mean ± std. Single-split numbers with `df` at n=9 will not survive a defence question about
variance.